# Sudoku Reloaded

Flujo completo: detección de recuadro de juego, recorte, detección de números y doble solución (CNN/Backtracking)

### Importaciones, comprobación GPU y gestión de Google Drive

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow import keras

gpus = tf.config.list_physical_devices('GPU')
print('✓ GPU disponible:', gpus if gpus else '❌ NO — ve a Entorno de ejecución → Cambiar tipo → GPU')

from google.colab import drive
drive.mount('/content/drive')

### Carga de CSV por bloques

Ya que el CSV de 1.3 GB daría muchos problemas al procesarlo de golpe, carga una cantidad máxima que también defino (en mi caso 1.000.000 de puzzles).

In [ ]:
RUTA_CSV = '/content/drive/MyDrive/Sudoku/sudoku.csv' # ruta desde donde se carga el CSV
MAX_PUZZLES = 1_000_000   # sudokus para entrenar

cabecera = pd.read_csv(RUTA_CSV, nrows=1) # lee solo la primera línea del archivo CSV para extraer
                                          # los nombres de los encabezados sin cargar datos pesados en memoria
cols = list(cabecera.columns) # convierte los encabezados en una lista y los imprime para verificar cómo se llaman
print('Columnas del CSV:', cols)

#Si encuentra textualmente 'puzzle' o 'solution', los usa; si el CSV tiene otros nombres,
# toma la primera y la segunda columna por posición
col_puzzle   = 'puzzle'   if 'puzzle'   in cols else cols[0]
col_solucion = 'solution' if 'solution' in cols else cols[1]
print(f'Usando: puzzle="{col_puzzle}", solución="{col_solucion}"')

# carga el DataFrame definitivo con el límite de registros establecido previamente
df = pd.read_csv(RUTA_CSV, nrows=MAX_PUZZLES)
print(f'✓ {len(df):,} puzzles cargados')

In [ ]:
# función que transforma cadenas de texto de 81 caracteres en arrays numéricos
def a_array(serie):
    # junta todas las filas del texto en una sola cadena gigante
    arr = np.frombuffer(''.join(serie.values).encode(), dtype=np.uint8) - ord('0')
    # redimensiona el array plano para darle una estructura bidimensional: una fila por cada sudoku, 
    # con 81 columnas correspondientes a las celdas
    return arr.reshape(len(serie), 81)

# procesa los sudokus pistas, convirtiendo los números a decimales
# y los divide entre 9.0 para normalizar las entradas en un rango de 0 a 1], facilitando la convergencia de la red
X = a_array(df[col_puzzle]).astype('float32') / 9.0   # normalizado 0-1
# procesa las soluciones, convirtiendo las respuestas a enteros pequeños (int8) y les resta 1.
# Esto transforma los números del sudoku, del 1 al 9, en índices de clase, del 0 al 8, que es el formato
# requerido por las funciones de pérdida de clasificación en Keras
y = a_array(df[col_solucion]).astype('int8') - 1       # clases 0-8

print(f'✓ X: {X.shape}  y: {y.shape}')
print(f'  Memoria X: {X.nbytes/1e6:.0f} MB, y: {y.nbytes/1e6:.0f} MB')

# elimina por completo el dataframe original de la memoria RAM y fuerza a Python 
# a liberar ese espacio de inmediato
del df
import gc; gc.collect()

### Definir la red neuronal convolucional para solucionar los sudokus

In [ ]:
modelo_sudoku = keras.Sequential([
    # capa de entrad, donde se toma el vector plano de 81 elementos de cada sudoku y lo transforma en una matriz bidimensional
    # de 9x9 con 1 canal de color, imitando el formato de una imagen en escala de grises para que las capas convolucionales
    # puedan interpretar la vecindad de las celdas
    keras.layers.Reshape((9, 9, 1), input_shape=(81,)),
    #capas intermedias
    keras.layers.Conv2D(64,  (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(64,  (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(256, (1,1), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    # capa de salida
    keras.layers.Conv2D(9, (1,1), activation='softmax', padding='same'),
    keras.layers.Reshape((81, 9))
])

modelo_sudoku.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001), # optimizador Adam con una tasa de aprendizaje inicial de 0,001
    loss='sparse_categorical_crossentropy', # función de coste
    metrics=['accuracy'] # métrica de rendimiento a monitorizar
)
modelo_sudoku.summary() # resumen de la red convolucional con el número total de parámetros y las dimensiones de salida de cada capa

### Entrenamiento

In [ ]:
# funciones automatizadas que controlan el estado del modelo al final de cada época del entrenamiento
callbacks = [
    # detiene el entrenamiento antes de tiempo si la precisión de validación deja de mejorar durante 4 épocas consecutivas
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    # si la pérdida de validación no disminuye durante 2 épocas, reduce la tasa de aprendizaje a la mitad
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, verbose=1),
    # guarda automáticamente el archivo del modelo en Google Drive cada vez que se detecta una mejora en val_accuracy
    keras.callbacks.ModelCheckpoint('/content/drive/MyDrive/Sudoku/modelo_sudoku_mejor.keras',
                                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

historia = modelo_sudoku.fit(
    X, y, # pasa las matrices de características de entrada normalizadas y las etiquetas correspondientes a sus soluciones
    epochs=30, # número máximo de pasadas completas sobre el conjunto de datos
    batch_size=256, # red para procesar los sudokus en grupos de 256 en 256 antes de actualizar los pesos
    validation_split=0.1, # separa un 10% de los datos de entrenamiento para evaluar el rendimiento del modelo
    callbacks=callbacks # aplica las funciones automatizadas
)

In [ ]:
# visualización de métricas
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(historia.history['accuracy'],     label='train')
axes[0].plot(historia.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy por celda'); axes[0].legend()
axes[1].plot(historia.history['loss'],     label='train')
axes[1].plot(historia.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()

acc = max(historia.history['val_accuracy'])
print(f'✓ Mejor accuracy por celda: {acc:.4f} ({acc*100:.2f}%)')

### Evaluación del modelo

In [ ]:
# extrae una muestra fija de los últimos 2000 registros del conjunto de datos para realizar una validación final
N_TEST = 2000
X_test = X[-N_TEST:]
y_test = y[-N_TEST:]

# hace predicciones sobre la muestra elegida, devolviendo una matriz con las distribuciones probabilísticas
pred = modelo_sudoku.predict(X_test, batch_size=256, verbose=1)
# evalúa los vectores probabilísticos 
pred_clases = np.argmax(pred, axis=2)   # (N, 81) con clases 0-8

# calcula el promedio global de aciertos comparando de manera individual cada una de las celdas frente al valor real esperado
acc_celda = np.mean(pred_clases == y_test)

# % de sudokus resueltos completos (todas las celdas correctas)
correctos_por_sudoku = np.all(pred_clases == y_test, axis=1)
acc_sudoku = np.mean(correctos_por_sudoku)

print(f'Accuracy por celda:    {acc_celda*100:.2f}%')
print(f'Sudokus 100% correctos: {acc_sudoku*100:.2f}%  ({correctos_por_sudoku.sum()}/{N_TEST})')
print('\n→ Para la presentación: la red sola resuelve perfecto el {:.1f}% de los sudokus.'.format(acc_sudoku*100))
print('  El backtracking corrige el resto garantizando el 100%.')

### Guardado del modelo final en Google Drive

In [ ]:
modelo_sudoku.save('/content/drive/MyDrive/Sudoku/modelo_sudoku.keras')
print('✓ Modelo guardado en Drive: modelo_sudoku.keras')